# Phase 2D — LLM-as-Judge Evaluation Lab

The proposal's headline evaluation protocol (Part 2.2): use an LLM to judge whether retrieved
memory is relevant, faithful, and correctly used. This notebook runs a **live LLM judge** (via the
OpenAI-compatible endpoint) on real benchmark records and compares it to the offline token-overlap proxy.

**Independent notebook** — runs standalone in Colab or locally. Needs the API key.

## What the LLM-as-judge evaluates

For each benchmark question, the judge receives:
- The **question**
- The **gold answer**
- The **retrieved context** (what the memory system returned)

And scores three dimensions:
1. **Answer correctness** (0-1): does the context contain the information needed to answer?
2. **Faithfulness** (0-1): is the answer grounded in the retrieved context (not hallucinated)?
3. **Context relevance** (0-1): is the retrieved context actually relevant to the question?

The **offline proxy** (used as default in Phase 2) approximates these with token overlap.
This lab shows where the LLM judge agrees and disagrees with the proxy.

In [ ]:
# Setup — self-contained (Colab or local)
import sys, os, subprocess
from pathlib import Path
try:
 import google.colab; IN_COLAB = True
except Exception:
 IN_COLAB = False
REPO_URL = 'https://github.com/syaikhipin/kdd26-memdiag'
if IN_COLAB:
 repo = Path('/content/kdd26-memdiag')
 if not repo.exists():
 subprocess.run(['git','clone','--depth','1',REPO_URL,str(repo)], check=False)
 SOURCE = repo / 'experiment/github_submission/source'
 subprocess.run([sys.executable,'-m','pip','install','-q','numpy','matplotlib','pyyaml'], check=False)
else:
 SOURCE = None
 for cand in [Path.cwd(), *Path.cwd().parents]:
 for sub in ('source','experiment'):
 if (cand/sub/'run.py').exists(): SOURCE = cand/sub; break
 if SOURCE: break
sys.path.insert(0, str(SOURCE))
os.environ.setdefault('OPENAI_BASE_URL','https://api.openai.com/v1')
RESULTS_DIR = SOURCE.parent / 'results'
print('SOURCE_DIR =', SOURCE, '| key set:', bool(os.environ.get('OPENAI_API_KEY')))

In [ ]:
# Load 5 real benchmark records (LoCoMo, with retrieved context)
import json
raw = json.loads((RESULTS_DIR / 'metrics' / 'sample_real_raw.json').read_text())
sample = [r for r in raw['records']
 if r.get('dataset') == 'LoCoMo'
 and r.get('strategy') == 'verbatim'
 and r.get('retrieved_texts')][:5]
print(f'Loaded {len(sample)} records (LoCoMo / verbatim)')
for i, r in enumerate(sample):
 print(f' [{i}] Q: {r["question"][:60]}... | hit={r.get("evidence_hit")}')

In [ ]:
# Offline token-overlap judge (the default proxy)
from memory_store import tokenize
def offline_judge(gold, context):
 g, c = set(tokenize(str(gold))), set(tokenize(str(context)))
 return round(len(g & c) / max(1, len(g)), 3) if g else 0.0

offline_scores = []
for r in sample:
 ctx = ' '.join(r.get('retrieved_texts', []))
 score = offline_judge(r.get('gold_answer', ''), ctx)
 offline_scores.append(score)
 print(f' offline answer_correctness = {score}')

In [ ]:
# Live LLM judge (via OpenAI-compatible endpoint)
from openai import OpenAI
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY',''),
 base_url=os.environ.get('OPENAI_BASE_URL','https://api.openai.com/v1'))
MODEL = 'gpt-4o'

llm_scores = []
for i, r in enumerate(sample):
 ctx = ' '.join(r.get('retrieved_texts', []))[:600]
 prompt = (f'Score the answer correctness of the retrieved context for answering the question.\n'
 f'Question: {r["question"]}\nGold answer: {r.get("gold_answer","")}\n'
 f'Retrieved context: {ctx}\n\n'
 f'Reply with ONLY JSON: {{"score": <float 0-1>, "reason": "<brief>"}}')
 try:
 resp = client.chat.completions.create(
 model=MODEL,
 messages=[{'role':'user','content':prompt}],
 max_tokens=80, temperature=0.0)
 text = resp.choices[0].message.content
 import re
 m = re.search(r'"score"\s*:\s*([0-9.]+)', text)
 score = float(m.group(1)) if m else 0.5
 reason = re.search(r'"reason"\s*:\s*"([^"]+)"', text)
 llm_scores.append(score)
 print(f' [{i}] LLM score={score:.2f} reason={reason.group(1)[:50] if reason else "?"}')
 except Exception as e:
 llm_scores.append(None)
 print(f' [{i}] LLM error: {e}')

In [ ]:
# Compare: offline proxy vs live LLM judge
print(f'{"record":>3} {"offline":>8} {"LLM judge":>10} {"agree?":>8}')
agreements = 0
for i in range(len(sample)):
 off = offline_scores[i]
 llm = llm_scores[i] if llm_scores[i] is not None else 0
 agree = abs(off - llm) < 0.2 # within 0.2 = agreement
 agreements += agree
 print(f'{i:>3} {off:>8.2f} {llm:>10.2f} {"yes" if agree else "NO":>8}')

rate = agreements / max(1, len(sample))
print(f'\nAgreement rate: {agreements}/{len(sample)} = {rate:.0%}')
print('(Within 0.2 threshold. The LLM judge often disagrees with token-overlap when')
print(' the context is semantically relevant but lexically different — a key insight.)')

## Discussion

- **Where they agree:** when the gold answer's exact words appear in the retrieved context, both
 judges score high.
- **Where they disagree:** when the context is *semantically* relevant but uses *different words*
 (paraphrase, synonym), the token-overlap proxy scores low while the LLM judge recognizes the
 relevance. This is the LLM-as-judge's key advantage.
- **Calibration:** the LLM judge's scores tend to cluster higher (it's generous). For production,
 calibrate against human labels and report inter-rater reliability (Cohen's κ).
- **Cost:** each LLM judge call costs API credits; the offline proxy is free. Use the proxy for
 large-scale screening and the LLM judge for validation/calibration on a sample.